# 01_setup.ipynb — PatchTST 環境確認

このノートブックでやること:
- 必要パッケージのインストール
- GPU確認
- PatchTST が正しく動くか動作確認（ダミーデータで推論まで）

## 1. パッケージインストール

Colab の VM にインストールされるので、ローカル環境は汚れません。  
セッションが切れると消えますが、次回実行時にまた `!pip install` するだけでOKです。

In [1]:
!pip install -q transformers==4.40.0 accelerate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 123.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


> **バージョン固定の理由**  
> `PatchTSTForPrediction` は transformers 4.37 以降で利用可能。  
> 最新版で API が変わる場合があるため 4.40.0 で固定しています。

## 2. GPU 確認

Colab メニュー → ランタイム → ランタイムのタイプを変更 → T4 GPU を選択してから実行してください。

In [2]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')

if device.type == 'cuda':
    print(f'GPU名: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')
else:
    print('⚠️ GPU が有効になっていません。ランタイムのタイプを確認してください。')

使用デバイス: cuda
GPU名: Tesla T4
VRAM: 15.6 GB


## 3. transformers バージョン確認

In [3]:
import transformers
print(f'transformers: {transformers.__version__}')

# PatchTST が import できるか確認
from transformers import PatchTSTConfig, PatchTSTForPrediction
print('PatchTSTForPrediction: OK')

transformers: 4.40.0
PatchTSTForPrediction: OK


## 4. PatchTST の構造を理解する

PatchTST（2023）のアイデア:

```
時系列 → patchに分割 → 各patchをtokenとしてTransformerに入力 → 予測

[─────────────── seq_len=512 ───────────────]
 [patch] [patch] [patch] ... [patch]   ← patch_length=16, stride=8
    ↓       ↓       ↓           ↓
 [token] [token] [token] ... [token]   ← Transformer Encoder
                                ↓
                         [予測: pred_len=7]
```

LSTMとの違い:
- LSTM: 時刻ごとに順番に処理（逐次）
- PatchTST: パッチをまとめて並列処理（Attention）→ GPU 活用率が高い

今回の設定:
- `context_length`（seq_len）= 30（過去30日）
- `prediction_length`（pred_len）= 7（翌7日）
- `patch_length` = 6（6日を1patch）
- `stride` = 1（1日ずつスライド）

## 5. ダミーデータで動作確認

実データを使う前に、PatchTSTが正しく動くかをダミーデータで確認します。

In [4]:
from transformers import PatchTSTConfig, PatchTSTForPrediction
import torch

# モデル設定
config = PatchTSTConfig(
    num_input_channels=1,      # 変数数（気温のみ = 1）
    context_length=30,          # 入力: 過去30日
    prediction_length=7,        # 出力: 翌7日
    patch_length=6,             # 1パッチ = 6日
    stride=1,                   # スライド幅
    d_model=128,                # Transformer の隠れ次元
    num_attention_heads=4,
    num_hidden_layers=3,
    dropout=0.2,
    head_dropout=0.2,
)

model = PatchTSTForPrediction(config).to(device)

# パラメータ数を確認
n_params = sum(p.numel() for p in model.parameters())
print(f'パラメータ数: {n_params:,}')

# モデル構造の確認
print(model)

パラメータ数: 599,815
PatchTSTForPrediction(
  (model): PatchTSTModel(
    (scaler): PatchTSTScaler(
      (scaler): PatchTSTStdScaler()
    )
    (patchifier): PatchTSTPatchify()
    (masking): Identity()
    (encoder): PatchTSTEncoder(
      (embedder): PatchTSTEmbedding(
        (input_embedding): Linear(in_features=6, out_features=128, bias=True)
      )
      (positional_encoder): PatchTSTPositionalEncoding(
        (positional_dropout): Identity()
      )
      (layers): ModuleList(
        (0-2): 3 x PatchTSTEncoderLayer(
          (self_attn): PatchTSTAttention(
            (k_proj): Linear(in_features=128, out_features=128, bias=True)
            (v_proj): Linear(in_features=128, out_features=128, bias=True)
            (q_proj): Linear(in_features=128, out_features=128, bias=True)
            (out_proj): Linear(in_features=128, out_features=128, bias=True)
          )
          (dropout_path1): Identity()
          (norm_sublayer1): PatchTSTBatchNorm(
            (batchnorm): Batch

In [5]:
# ダミーデータで推論テスト
batch_size = 4
dummy_input = torch.randn(batch_size, 30, 1).to(device)  # (batch, seq_len, channels)

model.eval()
with torch.no_grad():
    output = model(past_values=dummy_input)

print(f'入力 shape: {dummy_input.shape}')      # (4, 30, 1)
print(f'出力 shape: {output.prediction_outputs.shape}')  # (4, 7, 1)
print('推論テスト: OK ✅')

入力 shape: torch.Size([4, 30, 1])
出力 shape: torch.Size([4, 7, 1])
推論テスト: OK ✅


## 6. LSTM と PatchTST のパラメータ数比較

どちらが「重い」モデルか確認しておきます。

In [6]:
import torch.nn as nn

# 第3弾と同じ LSTM
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, pred_len=7):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, pred_len)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm = LSTMForecaster()
n_lstm = sum(p.numel() for p in lstm.parameters())

print(f'LSTM      パラメータ数: {n_lstm:,}')
print(f'PatchTST  パラメータ数: {n_params:,}')
print(f'比率: PatchTST は LSTM の {n_params / n_lstm:.1f} 倍')

LSTM      パラメータ数: 50,887
PatchTST  パラメータ数: 599,815
比率: PatchTST は LSTM の 11.8 倍


## 確認チェックリスト

- [ ] GPU が `cuda` と表示された
- [ ] `PatchTSTForPrediction: OK` が表示された
- [ ] 推論テストで出力 shape が `(4, 7, 1)` になった
- [ ] LSTM と PatchTST のパラメータ数を確認した

すべてOKなら `02_dataset.ipynb` へ進んでください。